# 🎯 Modelos Baseline

**Tech Challenge Fase 1 · Churn Predictor**

Antes de partir para a MLP, estabelecemos baselines em sklearn:
1. **DummyClassifier** — piso de performance
2. **LogisticRegression** — baseline linear interpretável
3. **RandomForestClassifier** — baseline não-linear
4. **GradientBoostingClassifier** — estado-da-arte tabular tradicional

Todos com validação cruzada estratificada e tracking no MLflow.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import StratifiedKFold

from churn_predictor.data.loader import load_and_split
from churn_predictor.evaluation.metrics import compute_metrics, compute_business_cost
from churn_predictor.features.pipeline import build_preprocessing_pipeline
from churn_predictor.models.baselines import get_baseline_models
from churn_predictor.utils.config import settings
from churn_predictor.utils.seeds import set_seeds

set_seeds(settings.random_seed)
sns.set_style("whitegrid")

## 1. Setup MLflow

In [ ]:
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment("churn-prediction-baselines-nb")

## 2. Carregamento e split dos dados

In [ ]:
split, metadata = load_and_split()
print("Metadata:", metadata)
print("\nSplit summary:")
for k, v in split.summary().items():
    print(f"  {k}: {v}")

## 3. Pipeline de pré-processamento

In [ ]:
pipeline = build_preprocessing_pipeline()
X_train = pipeline.fit_transform(split.X_train)
X_val = pipeline.transform(split.X_val)
X_test = pipeline.transform(split.X_test)
y_train = split.y_train.values
y_val = split.y_val.values
y_test = split.y_test.values

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

## 4. Treinamento dos baselines com CV

In [ ]:
results = []
trained_models = {}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=settings.random_seed)

for name, model in get_baseline_models().items():
    with mlflow.start_run(run_name=f"baseline_{name}"):
        # CV
        cv_scores = []
        for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
            model_cv = type(model)(**model.get_params())
            model_cv.fit(X_train[tr_idx], y_train[tr_idx])
            proba = model_cv.predict_proba(X_train[va_idx])[:, 1]
            m = compute_metrics(y_train[va_idx], proba)
            cv_scores.append(m.pr_auc)

        # Treina no full train
        model.fit(X_train, y_train)
        trained_models[name] = model

        # Avalia no test
        test_proba = model.predict_proba(X_test)[:, 1]
        test_metrics = compute_metrics(y_test, test_proba)
        cost = compute_business_cost(y_test, (test_proba >= 0.5).astype(int))

        results.append({
            "model": name,
            "cv_pr_auc_mean": np.mean(cv_scores),
            "cv_pr_auc_std": np.std(cv_scores),
            "test_pr_auc": test_metrics.pr_auc,
            "test_roc_auc": test_metrics.roc_auc,
            "test_f1": test_metrics.f1,
            "test_recall": test_metrics.recall,
            "test_precision": test_metrics.precision,
            "business_cost": cost["total_cost"],
        })

        mlflow.log_metric("cv_pr_auc_mean", np.mean(cv_scores))
        mlflow.log_metrics({f"test_{k}": v for k, v in test_metrics.to_dict().items() if isinstance(v, float)})
        mlflow.sklearn.log_model(model, name="model")

        print(f"✓ {name}: CV PR-AUC = {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f} | Test PR-AUC = {test_metrics.pr_auc:.4f}")

results_df = pd.DataFrame(results).sort_values("test_pr_auc", ascending=False)
results_df

## 5. Comparação visual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR-AUC e ROC-AUC
results_df_plot = results_df.set_index("model")[["test_pr_auc", "test_roc_auc", "test_f1"]]
results_df_plot.plot(kind="bar", ax=axes[0])
axes[0].set_title("Métricas no test set")
axes[0].set_ylabel("Score")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(loc="lower right")

# Custo de negócio
results_df.set_index("model")["business_cost"].plot(kind="bar", ax=axes[1], color="#e74c3c")
axes[1].set_title("Custo total de negócio (R$)")
axes[1].set_ylabel("Custo (R$)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 6. Conclusão dos baselines

- **Dummy** confirma que o problema tem sinal — qualquer modelo razoável supera ~0.27 PR-AUC.
- **Logistic Regression** entrega ~0.65 PR-AUC com alta interpretabilidade — bom candidato a fallback.
- **Gradient Boosting** lidera entre baselines com ~0.66 PR-AUC.
- **Random Forest** fica próximo, mas com recall menor.

**Próximo passo:** treinar a MLP em PyTorch e ver se conseguimos ultrapassar o Gradient Boosting (notebook `03_mlp.ipynb`).